# Exploratory Data Analysis: PISA 2022 Germany

Treatment: `PRESCHOOL`  
Outcome: `READ`

In [ ]:
# Import packages

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

RANDOM_STATE = 123

## 1. Load data and define variables

In [ ]:
# Load data

file = "PISA2022deu.csv"
data_raw = pd.read_csv(file)

# Recode treatment

data_raw["PREEDU"] = pd.to_numeric(data_raw["PREEDU"], errors="coerce")
data_raw["PRESCHOOL"] = 1 - data_raw["PREEDU"]

print("Raw data:")
print(f"Rows:    {data_raw.shape[0]}")
print(f"Columns: {data_raw.shape[1]}")

display(data_raw.head())

In [ ]:
# Define variables

treatment = "PRESCHOOL"
outcome = "READ"

covariates = [
    "GENDER",
    "AGE",
    "IMMIG",
    "HISCED",
    "HOMEPOS",
    "ESCS",
    "CREATHME",
    "PA188Q03JA",
    "PA005Q01TA",
    "HISEI",
]

analysis_vars = [treatment, outcome] + covariates

missing_columns = [v for v in analysis_vars if v not in data_raw.columns]
if missing_columns:
    raise ValueError(f"Missing variables in the data: {missing_columns}")

data = data_raw[analysis_vars].copy()

overview = pd.DataFrame({
    "Variable": analysis_vars,
    "Role": ["Treatment", "Outcome"] + ["Covariate"] * len(covariates),
    "Dtype": [str(data[v].dtype) for v in analysis_vars],
    "Unique values": [data[v].nunique(dropna=True) for v in analysis_vars],
})

display(overview)

## 2. Missing values and complete cases

In [ ]:
# Missing-value table

missing_table = pd.DataFrame({
    "Missing N": data.isna().sum(),
    "Missing %": data.isna().mean() * 100,
    "Non-missing N": data.notna().sum()
}).sort_values("Missing %", ascending=False)

display(missing_table.round(2))

In [ ]:
# Missing-value plot

missing_plot = missing_table[missing_table["Missing N"] > 0].copy()

plt.figure(figsize=(9, 4))
if len(missing_plot) > 0:
    plt.bar(missing_plot.index, missing_plot["Missing %"])
    plt.ylabel("Missing values (%)")
    plt.title("Missing values by analysis variable")
    plt.xticks(rotation=45, ha="right")
else:
    plt.text(0.5, 0.5, "No missing values in the analysis variables", ha="center", va="center")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Complete-case sample

data_cc = data.dropna().copy()

print("Complete-case sample:")
print(f"Rows before: {data.shape[0]}")
print(f"Rows after:  {data_cc.shape[0]}")
print(f"Dropped:     {data.shape[0] - data_cc.shape[0]} ({(1 - data_cc.shape[0] / data.shape[0]) * 100:.2f}%)")

display(data_cc.head())

## 3. Treatment distribution

In [ ]:
# Treatment frequencies

preschool_counts = data_cc[treatment].value_counts(dropna=False).sort_index()
preschool_perc = data_cc[treatment].value_counts(normalize=True, dropna=False).sort_index() * 100

preschool_table = pd.DataFrame({
    "N": preschool_counts,
    "%": preschool_perc
})

display(preschool_table.round(2))

In [ ]:
# Treatment labels

TREATMENT_LABELS = {
    0: "No preschool",
    1: "Preschool",
    "0": "No preschool",
    "1": "Preschool",
}

data_cc["PRESCHOOL_label"] = data_cc[treatment].map(TREATMENT_LABELS)
data_cc["PRESCHOOL_label"] = data_cc["PRESCHOOL_label"].fillna(data_cc[treatment].astype(str))

display(data_cc[[treatment, "PRESCHOOL_label"]].drop_duplicates().sort_values(treatment))

In [ ]:
# Treatment distribution plot

count_by_label = data_cc["PRESCHOOL_label"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(count_by_label.index.astype(str), count_by_label.values)
plt.ylabel("Number of students")
plt.title("Distribution of PRESCHOOL groups")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Outcome distribution and group differences

In [ ]:
# Outcome summary

read_desc = data_cc[outcome].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame(name=outcome)
display(read_desc.round(2))

In [ ]:
# Outcome histogram

plt.figure(figsize=(7, 4))
plt.hist(data_cc[outcome], bins=30, edgecolor="black")
plt.xlabel("READ")
plt.ylabel("Frequency")
plt.title("Distribution of reading performance")
plt.tight_layout()
plt.show()

In [ ]:
# Outcome by treatment group

read_by_preschool = data_cc.groupby("PRESCHOOL_label")[outcome].agg(
    N="count",
    Mean="mean",
    SD="std",
    Median="median",
    Min="min",
    Max="max"
).sort_index()

display(read_by_preschool.round(2))

In [ ]:
# Mean difference, t-test, and Cohen's d

groups = []
for label, subset in data_cc.groupby("PRESCHOOL_label"):
    groups.append((label, subset[outcome].dropna()))

if len(groups) == 2:
    label_a, y_a = groups[0]
    label_b, y_b = groups[1]

    t_stat, p_val = stats.ttest_ind(y_a, y_b, equal_var=False)

    pooled_sd = np.sqrt(((len(y_a) - 1) * y_a.var(ddof=1) + (len(y_b) - 1) * y_b.var(ddof=1)) / (len(y_a) + len(y_b) - 2))
    cohen_d = (y_b.mean() - y_a.mean()) / pooled_sd

    test_table = pd.DataFrame({
        "Comparison": [f"{label_b} minus {label_a}"],
        "Mean difference": [y_b.mean() - y_a.mean()],
        "t": [t_stat],
        "p": [p_val],
        "Cohen d": [cohen_d],
    })
    display(test_table.round(4))
else:
    print("The t-test is displayed only if exactly two PRESCHOOL groups are present.")

In [ ]:
# Outcome boxplot by treatment group

labels = list(read_by_preschool.index)
data_to_plot = [data_cc.loc[data_cc["PRESCHOOL_label"] == label, outcome] for label in labels]

plt.figure(figsize=(7, 4))
plt.boxplot(data_to_plot, labels=labels, showmeans=True)
plt.ylabel("READ")
plt.title("Reading performance by PRESCHOOL group")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Outcome histograms by treatment group

plt.figure(figsize=(8, 4))
for label, subset in data_cc.groupby("PRESCHOOL_label"):
    plt.hist(subset[outcome], bins=25, alpha=0.5, label=str(label), edgecolor="black")

plt.xlabel("READ")
plt.ylabel("Frequency")
plt.title("Distribution of READ by PRESCHOOL group")
plt.legend(title="PRESCHOOL")
plt.tight_layout()
plt.show()

## 5. Covariate summaries

In [ ]:
# Detect numeric and categorical covariates

def to_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

numeric_candidates = []
categorical_candidates = []

for v in covariates:
    s_num = to_numeric_series(data_cc[v])
    share_numeric = s_num.notna().mean()

    if share_numeric > 0.95:
        numeric_candidates.append(v)
        data_cc[v + "_num"] = s_num
    else:
        categorical_candidates.append(v)

print("Numeric covariates:")
print(numeric_candidates)

print("\nCategorical covariates:")
print(categorical_candidates)

In [ ]:
# Numeric covariate summary

num_cols = [v + "_num" for v in numeric_candidates]

if len(num_cols) > 0:
    desc_num = data_cc[num_cols].describe().T
    desc_num.index = numeric_candidates
    display(desc_num.round(2))
else:
    print("No numeric covariates detected.")

In [ ]:
# Numeric covariates by treatment group

if len(num_cols) > 0:
    grouped_num = data_cc.groupby("PRESCHOOL_label")[num_cols].agg(["mean", "std", "count"])
    grouped_num.columns = [f"{col.replace('_num', '')}_{stat}" for col, stat in grouped_num.columns]
    display(grouped_num.round(2))
else:
    print("No numeric covariates detected.")

In [ ]:
# Frequency tables for low-cardinality covariates

low_cardinality_vars = []
for v in covariates:
    if data_cc[v].nunique(dropna=True) <= 10:
        low_cardinality_vars.append(v)

for v in low_cardinality_vars:
    print("\n" + "=" * 80)
    print(f"Frequencies: {v}")
    tab = pd.crosstab(data_cc[v], data_cc["PRESCHOOL_label"], margins=True)
    display(tab)

    print(f"Column percentages: {v}")
    tab_pct = pd.crosstab(data_cc[v], data_cc["PRESCHOOL_label"], normalize="columns") * 100
    display(tab_pct.round(2))

## 6. Covariate balance

In [ ]:
# Balance table for numeric covariates

def standardized_mean_difference(x0, x1):
    pooled_sd = np.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan
    return (x1.mean() - x0.mean()) / pooled_sd

unique_groups = list(data_cc["PRESCHOOL_label"].dropna().unique())

if len(unique_groups) == 2 and len(num_cols) > 0:
    g0, g1 = sorted(unique_groups)
    balance_rows = []

    for v, v_num in zip(numeric_candidates, num_cols):
        x0 = data_cc.loc[data_cc["PRESCHOOL_label"] == g0, v_num].dropna()
        x1 = data_cc.loc[data_cc["PRESCHOOL_label"] == g1, v_num].dropna()
        t_stat, p_val = stats.ttest_ind(x0, x1, equal_var=False)

        balance_rows.append({
            "Variable": v,
            f"Mean {g0}": x0.mean(),
            f"Mean {g1}": x1.mean(),
            f"Difference {g1} - {g0}": x1.mean() - x0.mean(),
            "Std. mean difference": standardized_mean_difference(x0, x1),
            "p": p_val,
        })

    balance_table = pd.DataFrame(balance_rows).sort_values("Std. mean difference", key=lambda s: s.abs(), ascending=False)
    display(balance_table.round(4))
else:
    print("Balance table is displayed only if exactly two treatment groups and numeric covariates are present.")

In [ ]:
# Balance plot

if "balance_table" in globals():
    bt = balance_table.copy().sort_values("Std. mean difference")

    plt.figure(figsize=(8, 5))
    plt.barh(bt["Variable"], bt["Std. mean difference"])
    plt.axvline(0, linestyle="--")
    plt.axvline(0.1, linestyle=":")
    plt.axvline(-0.1, linestyle=":")
    plt.xlabel("Standardized mean difference")
    plt.title("Covariate balance between PRESCHOOL groups")
    plt.tight_layout()
    plt.show()
else:
    print("No balance table available.")

## 7. Correlations

In [ ]:
# Numeric correlation matrix

corr_vars = [outcome, treatment] + numeric_candidates
corr_data = data_cc[corr_vars].copy()

for v in corr_vars:
    corr_data[v] = pd.to_numeric(corr_data[v], errors="coerce")

corr_numeric = corr_data.corr()

display(corr_numeric.round(3))

In [ ]:
# Correlation heatmap

plt.figure(figsize=(9, 7))
im = plt.imshow(corr_numeric, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr_numeric.columns)), corr_numeric.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_numeric.index)), corr_numeric.index)
plt.title("Correlation matrix of numeric variables")

for i in range(len(corr_numeric.index)):
    for j in range(len(corr_numeric.columns)):
        val = corr_numeric.iloc[i, j]
        if not np.isnan(val):
            plt.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Dummy-coded correlations

X_for_corr = data_cc[analysis_vars].copy()

for col in X_for_corr.columns:
    s_num = pd.to_numeric(X_for_corr[col], errors="coerce")
    if s_num.notna().mean() > 0.95:
        X_for_corr[col] = s_num

X_dummies = pd.get_dummies(X_for_corr, drop_first=True)

corr_full = X_dummies.corr()

focus_cols = [c for c in corr_full.columns if c == outcome or c == treatment or c.startswith(treatment + "_")]
focus_corr = corr_full[focus_cols].drop(index=focus_cols, errors="ignore").sort_values(focus_cols[0], key=lambda s: s.abs(), ascending=False)

display(focus_corr.round(3).head(30))

In [ ]:
# Top correlations with READ

corr_with_read = corr_full[outcome].drop(labels=[outcome], errors="ignore").sort_values(key=lambda s: s.abs(), ascending=False)
top_corr_read = corr_with_read.head(15).sort_values()

plt.figure(figsize=(8, 6))
plt.barh(top_corr_read.index, top_corr_read.values)
plt.axvline(0, linestyle="--")
plt.xlabel("Correlation with READ")
plt.title("Strongest bivariate associations with reading performance")
plt.tight_layout()
plt.show()

## 8. Visual associations with READ

In [ ]:
# Scatterplots for selected numeric covariates

plot_candidates = [v for v in ["ESCS", "HOMEPOS", "HISEI", "AGE", "CREATHME"] if v in data_cc.columns]

for v in plot_candidates:
    x = pd.to_numeric(data_cc[v], errors="coerce")
    y = pd.to_numeric(data_cc[outcome], errors="coerce")
    valid = x.notna() & y.notna()

    plt.figure(figsize=(6, 4))
    plt.scatter(x[valid], y[valid], alpha=0.35)

    if valid.sum() > 2:
        coef = np.polyfit(x[valid], y[valid], 1)
        x_line = np.linspace(x[valid].min(), x[valid].max(), 100)
        y_line = coef[0] * x_line + coef[1]
        plt.plot(x_line, y_line)

    plt.xlabel(v)
    plt.ylabel("READ")
    plt.title(f"Association between {v} and READ")
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots for selected categorical covariates

boxplot_candidates = [v for v in ["GENDER", "IMMIG", "HISCED", "PA188Q03JA", "PA005Q01TA"] if v in data_cc.columns]

for v in boxplot_candidates:
    if data_cc[v].nunique(dropna=True) <= 12:
        categories = sorted(data_cc[v].dropna().unique())
        values = [data_cc.loc[data_cc[v] == cat, outcome].dropna() for cat in categories]

        plt.figure(figsize=(8, 4))
        plt.boxplot(values, labels=[str(c) for c in categories], showmeans=True)
        plt.xlabel(v)
        plt.ylabel("READ")
        plt.title(f"READ by {v}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 9. Output tables

In [ ]:
# Export tables

missing_table.to_csv("eda_missing_table_preschool_read.csv")

if "read_by_preschool" in globals():
    read_by_preschool.to_csv("eda_read_by_preschool.csv")

if "balance_table" in globals():
    balance_table.to_csv("eda_balance_table_preschool_read.csv", index=False)

corr_numeric.to_csv("eda_correlation_numeric_preschool_read.csv")

print("Saved output tables.")